# Chapter 7: Search In-Depth
## Topic 2: Dense Retrieval — Recap and Its Specific Weaknesses

**One notebook. Theory + Code together.**
- Part A: Theory (Concept -> Revision Summary)
- Part B: Code (modules, run top to bottom)


## Part A: Theory

### 1. Concept, Intuition, and Why It Exists

- Dense retrieval maps every piece of text — a query, a document chunk, a policy clause — into a single fixed-size list of numbers (a vector).
- This mapping is learned by a model trained on millions of text pairs, so texts with similar meaning end up as vectors pointing in nearly the same direction, and texts with different meaning point in different directions.
- "Similar meaning" becomes "small angle between vectors" — something we can compute with simple math (cosine similarity).
- This is the one thing BM25 fundamentally cannot do: BM25 has no idea that "exit FD early" and "premature withdrawal" mean the same thing, because it only looks at matching words. A dense embedding model can recognize this.

**Why this exists — the one gap BM25 leaves open:**

- BM25's score is exactly 0 when there is zero vocabulary overlap between query and document.
- A customer writing "mera paisa fasa hua hai" (my money is stuck) gets a BM25 score of 0 against a policy chunk saying "premature closure penalty" — not a single word overlaps.
- Dense retrieval gives a meaningful similarity score for this pair, because both texts describe the same underlying situation (money being held, restricted access), regardless of the exact words or language used.
- This is not a rare edge case in this project: 64.4% of the 2,500 emails are Hinglish, and the policy documents are in English — vocabulary mismatch is the dominant failure mode, not an exception.

**This is not new to the project:**

- Chapter 3's embedding/semantic-search work already built this idea, using `paraphrase-multilingual-MiniLM-L12-v2` (384 dimensions) running locally on CPU.
- The pattern: `model.encode(texts, normalize_embeddings=True)` for batch embedding, then `np.dot(a, b)` for cosine similarity (equal to dot product for normalized vectors).
- The in-memory version from Chapter 3 is brute-force dense retrieval: embed everything once, embed the query, compare the query against every stored vector, return the top-k closest ones.


### 2. Internal Working — Step by Step

**Step 1: What happens inside the embedding model**

- `paraphrase-multilingual-MiniLM-L12-v2` is a BERT-style encoder with 12 transformer layers.
- Input text is broken into subword tokens, each token gets a position embedding.
- The 12 layers apply bidirectional self-attention — every token looks at every other token in both directions at once (this is different from GPT-style models, which only look backward).
- Output: one 384-number vector per token in the input.
- Pooling step: average all the token vectors together into a single 384-number vector representing the whole input — this is called mean pooling.
- That single averaged vector is the embedding — it represents the overall meaning of the text.

**Step 2: Normalization and why it matters**

- `model.encode(texts, normalize_embeddings=True)` rescales every output vector so its length is exactly 1 (this is called unit length, or L2-normalized).
- Once vectors have length 1, cosine similarity becomes mathematically identical to a simple dot product — cheaper to compute and simpler to reason about.
- Qdrant (used in Chapter 6) relies on this same shortcut internally when the distance metric is set to cosine.

**Step 3: Offline index building (happens once, at ingest time)**

- Every chunk from the knowledge base goes through `model.encode()` once.
- The resulting vector, along with the original text and metadata, gets stored (e.g., upserted into Qdrant).
- Qdrant builds and maintains an HNSW index (Chapter 6, Topic 4) over all these stored vectors.
- This only needs to happen again when new documents are added — not on every query.

**Step 4: Query embedding (happens at query time)**

- The customer's email or rewritten query goes through the exact same model used at ingest time.
- This must be the same model — using a different model at query time produces vectors in a different, incompatible vector space, and similarity scores become meaningless.

**Step 5: Approximate nearest-neighbor search**

- The query vector is compared against the stored vectors using the HNSW index.
- "Closest" means smallest angle, which for normalized vectors means largest dot product.
- The system returns the top-k closest chunks along with their similarity scores.
- "Approximate" means the index might occasionally miss the single mathematically closest vector — acceptable for RAG, since the next-closest chunk is almost always still useful.

**How to read the similarity scores:**

- A score of 1.0 means the vectors are identical (essentially the same text).
- Around 0.7-0.9 means strong semantic overlap.
- Around 0.5-0.7 means moderate overlap.
- Around 0.4-0.5 means weak overlap — this is where this project's specific weakness lives.
- Below 0.3 means little to no semantic relationship.


### 3. The Measured Weakness — Real Numbers From This Project

- Two same-topic FD complaints, written differently, scored a cosine similarity of 0.4528.
- One of those same FD complaints compared against a completely unrelated, Non-FD email scored 0.4135.
- The gap between "same topic, different words" and "totally different topic" is only 0.4528 - 0.4135 = 0.0393 — less than 4 percentage points.
- A retrieval system returning the top-5 chunks needs to reliably separate relevant from irrelevant content by at least this margin — and 0.0393 is dangerously thin.
- For comparison, an older Word2Vec-based check on the same kind of emails showed a gap of about 0.09 (more than double), even though Word2Vec is a much simpler, older technique.
- This is counterintuitive at first — isn't the newer sentence-transformer model supposed to be better? Yes, but the issue isn't model quality, it's input quality: short (~31-word), ambiguous, Hinglish emails simply don't carry enough signal for any embedding model to discriminate perfectly.
- This is the core, measured reason dense retrieval alone is not enough for this corpus, and why Hybrid Search (Topic 4) exists.


### 4. How This Is Implemented in This Project

- Chapter 3 implements this as brute-force, in-memory dense retrieval: embed the entire knowledge base once, embed each incoming query, compute cosine similarity against every stored vector, sort, and return the top-k.
- Chapter 6 upgrades this to Qdrant: vectors are stored persistently, and search uses the HNSW index (instead of comparing against every vector one by one), which scales far better as the knowledge base grows.
- The critical implementation rule in both versions: the exact same embedding model must be used at ingest time and query time, or the vectors won't be comparable at all.
- The code in Part B below rebuilds the core ideas offline, since the real `paraphrase-multilingual-MiniLM-L12-v2` model requires downloading weights from the internet. Instead, we build actual dense vectors using TF-IDF + SVD (a real, classic dense-vector technique called Latent Semantic Analysis) so every concept — vectors, normalization, cosine similarity, the discrimination-gap problem — can be demonstrated with real, executable numbers.


### 5. Real-World Issues, Edge Cases, Debugging, Monitoring, Scaling, Latency, Cost, Security, Deployment

- **Exact identifiers fail under dense retrieval:** an FD reference number like "BJ2019FD7717" gets broken into subword pieces during tokenization. A different customer's reference number, like "BJ2021FD4432", shares some of those same subword pieces ("BJ", "FD"), so the two numbers can embed to similar vectors even though they refer to completely different accounts. This risks cross-customer data leakage if used for lookups. The correct fix: use exact-match database lookup for identifiers, never semantic search.
- **Novel product names fail too:** a newly launched product with a name the embedding model has never seen during training gets a weak, imprecise embedding. BM25 can still find it by exact term match — this is exactly why hybrid search matters.
- **Cost:** running the embedding model locally is essentially free per query (just compute), unlike API-based embedding services that charge per token. For roughly 8,000-12,000 emails/day at ~31 words each, this is a small but real ongoing cost if using an API, versus a one-time compute cost if run locally.
- **Latency:** on CPU, embedding a batch of texts takes tens of milliseconds; on a local GPU (like an RTX 4060), this drops to single-digit milliseconds. Batch processing multiple texts together is significantly faster than embedding them one at a time in a loop.
- **Memory:** each 384-dimension vector takes a small, fixed amount of memory. At small scale (a handful of source documents), this is negligible; even at 100,000 chunks it remains manageable.
- **Monitoring:** track the distribution of top-1 similarity scores over time — a downward drift signals retrieval quality degrading. Also track how often the top-1 and top-2 scores are very close together — a high fraction of "too close to call" results signals genuine retrieval uncertainty, not just noise.
- **Security:** embedding vectors can, in theory, leak information about the original text (this is an active area of security research called vector inversion). If the original text contains PII (like account numbers), storing its embedding is effectively storing a secondary, indirect copy of that PII — apply the same access controls to the vector store as you would to the raw data. Running the embedding model locally also avoids sending any customer data to an external API.
- **Deployment:** for moderate daily volume, a single process with the model loaded once at startup on CPU is sufficient — no GPU required. For strict low-latency requirements, move to GPU and batch incoming queries together for efficiency.


### 6. Design Decisions, Trade-offs, and Real-Time Dilemmas

- **Why a multilingual model specifically:** the project's data mixes English (policy documents) and Hinglish (customer emails) — a model trained only on English would badly misrepresent the Hinglish half of the corpus.
- **Why mean pooling instead of using a single special token's output:** mean pooling averages the contribution of every word, rather than relying on one summary token, and tends to produce better general-purpose sentence embeddings for retrieval tasks. A special summary token can work well when a model has been fine-tuned for a specific task (like classification), but for general semantic search without task-specific fine-tuning, mean pooling is the safer default.
- **Brute-force in-memory search vs an indexed vector database:** brute-force guarantees you always find the true best match and is simple to reason about, but its search time grows linearly with the number of documents. An indexed approach (like HNSW in Qdrant) scales far better as the knowledge base grows, at the cost of occasionally missing the single best match (an acceptable trade-off for RAG). At a handful of documents both are equally fast; at tens of thousands of documents, indexing becomes necessary.
- **Should the embedding model be fine-tuned on domain-specific text?** Fine-tuning could genuinely help align Hinglish domain terms with their English equivalents in vector space, but it requires labeled training pairs, which are expensive to create and maintain. The practical trigger for considering this: only after measuring that hybrid search plus query expansion plus reranking are still insufficient — fine-tuning is a later-stage lever, not a first fix.


### 7. Alternatives and When to Use Each

- **Local multilingual sentence-transformer models (this project's actual choice):** free, fast, runs offline, and covers both English and Hinglish in the same model — the right choice for this corpus at this stage.
- **English-only local models:** faster and lighter, but would badly misrepresent the Hinglish portion of the corpus — wrong choice here despite being technically simpler.
- **Larger local multilingual models:** higher quality, meaningfully slower — worth evaluating only if the current model's already-thin discrimination gap proves insufficient even after hybrid search and reranking are added.
- **API-based embedding models:** can offer stronger quality at low per-token cost, but send text to an external service and introduce a small ongoing operating cost — worth evaluating as a quality upgrade path, not a starting point.
- **Specialized retrieval architectures (DPR, ColBERT — covered in Topic 3):** designed specifically for retrieval rather than adapted from a general sentence-embedding model, often better suited to genuinely large-scale or precision-critical retrieval systems.

### 8. Common Mistakes and Production Failures

- Using different embedding models at ingest time and query time — the most damaging failure, because retrieval silently returns meaningless results with no error message. It usually shows up only through monitoring, as a sudden unexplained drop in similarity scores.
- Forgetting to normalize vectors — without normalization, longer pieces of text naturally produce longer vectors, which score artificially higher regardless of actual relevance.
- Embedding texts one at a time in a loop instead of in a batch — this is dramatically slower than it needs to be, since these models are built to process many inputs at once efficiently.
- Using dense retrieval for exact-match lookups like account or reference numbers — this returns similar-looking but wrong records; exact identifiers need exact-match database lookup, not semantic similarity.
- Never monitoring the similarity score distribution — a thin discrimination gap like the one measured in this project means retrieval can be silently failing well before it becomes obvious in downstream answer quality.
- Assuming dense retrieval will always find brand-new terms it has never seen before — it won't; that's specifically what BM25 is good at, which is why the two are combined.
- Using an English-only embedding model on a majority-Hinglish corpus — this collapses all the Hinglish text toward whatever nearby English words happen to exist in the model's vocabulary, destroying meaningful distinctions.


### 9. Lead-Level Interview Questions

**Basic**

- Q: What is the core difference between sparse retrieval (BM25) and dense retrieval?
  A: BM25 matches literal words, weighted by rarity and frequency, and scores zero when there's no word overlap. Dense retrieval maps text to vectors in a learned space where semantic similarity corresponds to geometric closeness, so it can score positively even with zero word overlap, as long as the meanings are related.

- Q: Why normalize embedding vectors before comparing them?
  A: Normalizing to unit length makes cosine similarity mathematically identical to a simple dot product, which is cheaper to compute and ensures that longer texts don't score artificially higher just because their un-normalized vectors happen to be longer.

**Intermediate**

- Q: Two same-topic emails scored 0.4528 similarity, and one of them scored 0.4135 against a completely unrelated email. What does this tell you?
  A: The discrimination margin is only about 0.039 — dangerously thin. This means dense retrieval alone can't reliably separate relevant from irrelevant content in this corpus, threshold-based filtering will misfire in both directions, and combining dense retrieval with BM25 (hybrid search) is necessary rather than optional.

- Q: Why would semantic search fail for an exact identifier like an account reference number?
  A: The identifier gets broken into subword pieces during tokenization, and other identifiers sharing some of those same pieces can embed to similar vectors even though they refer to entirely different records. This risks returning someone else's data. Exact identifiers should always use exact-match lookup, not semantic similarity.

**Advanced**

- Q: The embedding model was trained on general multilingual text, not this specific domain. What retrieval failures does this create, and how would you address them?
  A: Domain-specific vocabulary may not be positioned close to its plain-English equivalent in vector space, since the model may not have seen enough domain-specific training examples. Code-mixed language (mixing two languages in one sentence) may embed ambiguously even if the model handles each language well individually. Brand-new terms coined after the model's training will have weak, uninformative embeddings. The practical fix order: measure hybrid BM25 + dense retrieval performance first, then add query expansion for known synonym pairs, and only consider fine-tuning the embedding model itself if that's still insufficient.

- Q: A colleague worries that using an approximate nearest-neighbor index means missing relevant documents that an exact brute-force search would find. How do you respond?
  A: The trade-off is real but the practical impact is small. Approximate indexes typically achieve very high recall at the result-list sizes used in RAG, and what they occasionally miss tends to be the second- or third-best match, not the best one. For most RAG use cases, the latency and scalability benefits far outweigh this small risk. If perfect exact guarantees were required — for something like fraud detection or legal matching — brute-force exact search would be justified instead.

**Scenario-based**

- Q: After deployment, answers about a newly launched product are consistently wrong or missing. How do you diagnose this?
  A: Start at the retrieval layer, not generation. First confirm the new product's documents were actually ingested into the knowledge base. If they were, check whether BM25 finds them by exact term match — it should. If BM25 finds them but dense retrieval doesn't, the cause is a weak embedding for a term the model has never seen before; the fix is to make sure hybrid search weights BM25 strongly enough that this kind of exact-match case isn't drowned out. If BM25 also fails to find them, the documents likely were never properly ingested in the first place.

**Follow-up questions to expect:**

- "If you could only keep one of BM25 or dense retrieval, which would you keep for this project, and what would you lose?"
- "How would you detect a thin discrimination-gap problem in production before it shows up as bad answers?"
- "What would change about your embedding strategy if the corpus were 100% English instead of majority Hinglish?"


### 10. Hidden Concepts / Prerequisites Worth Knowing

- **Mean pooling vs using a single summary token:** BERT-style models produce one vector per input token; to get a single vector for the whole sentence, you either average all token vectors (mean pooling) or use one designated summary token's output. Mean pooling tends to work better for general-purpose sentence embeddings used in retrieval, because it preserves signal from every word instead of relying on just one token.
- **The curse of dimensionality, and why it doesn't break dense retrieval:** in very high-dimensional spaces, distances between random points tend to become nearly equal, making distance a poor discriminator. 384 dimensions is fairly high-dimensional, yet dense retrieval still works well — because the embedding model isn't placing vectors randomly. It has learned a structured space where related meanings genuinely cluster together, so the "curse" that applies to random vectors doesn't apply here.
- **Why cosine similarity (angle) rather than raw distance (magnitude) is preferred for embeddings:** in learned embedding spaces, the direction of a vector carries the meaning, while its raw length often doesn't carry much useful signal. Comparing angles (cosine similarity) captures this better than comparing raw distances.
- **Bi-encoders vs cross-encoders:** the setup described here is a bi-encoder — the query and the document are embedded completely independently, and similarity is computed afterward. A cross-encoder instead looks at the query and document together as a single combined input and directly outputs one relevance score — this is typically far more accurate, because the model can see how the two texts interact, but it's also far slower, since it must be run separately for every candidate pair rather than compared with simple vector math. This bi-encoder/cross-encoder distinction is the foundation for reranking, covered later in this chapter.
- **Semantic similarity and lexical similarity are complementary, not competing:** BM25 measures how much the words overlap; dense retrieval measures how close the meanings are in a learned space. Neither alone captures both dimensions well, which is exactly the motivation for Hybrid Search.

### 11. Quick Revision Summary (for last-minute interview prep)

> Dense retrieval maps queries and documents into a learned vector space where distance reflects meaning, not just word overlap — the one thing BM25 fundamentally cannot do. This project's own measured data shows the real risk: the similarity gap between "same topic, different words" and "completely different topic" was only about 0.039 for short Hinglish emails, dangerously thin for reliable standalone retrieval. Dense retrieval also specifically struggles with exact identifiers (which should always use exact-match database lookup, never semantic search) and brand-new terms the model has never seen during training (which BM25 handles fine by exact match). The universal rules: always use the exact same embedding model at ingest and query time, always normalize vectors before comparing them, always batch-embed rather than looping one at a time, and never rely on dense retrieval alone for a corpus like this one — which is exactly why Hybrid Search (Topic 4) exists.


### Module 1: Setup — Building Real Dense Vectors Offline

The real `paraphrase-multilingual-MiniLM-L12-v2` model needs internet access to download its weights. Since that isn't available here, we build actual dense vectors using TF-IDF + SVD (Latent Semantic Analysis) — a real, classic dense-embedding technique. Every concept below (vectors, normalization, cosine similarity, the discrimination-gap problem) works the same way regardless of which model produced the vectors.

In [1]:

# ------------------------------------------------------------------
# MODULE 1: Setup -- offline dense vectors via TF-IDF + SVD
#
# Why TF-IDF + SVD counts as "real" dense retrieval:
#   TF-IDF gives us a SPARSE vector (mostly zeros, one dim per word).
#   SVD (Singular Value Decomposition) compresses that sparse vector
#   into a small, DENSE vector (every dimension is non-zero) that
#   still captures the main patterns of word co-occurrence.
#   This is literally how the earliest "dense embeddings" (LSA, 1990s)
#   worked, before neural network embeddings existed.
# ------------------------------------------------------------------

import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD

# a small knowledge base -- same style of content as the real project's
# FD policy / FAQ / SOP chunks
DENSE_KNOWLEDGE_BASE = [
    "Premature withdrawal of FD incurs a 1 percent penalty on the applicable interest rate.",
    "Fixed Deposit premature closure is allowed subject to applicable penalty charges as per RBI guidelines.",
    "Early exit from your deposit account will attract a deduction from your returns.",   # same MEANING as above, different words
    "The Fixed Deposit interest rate for 24 months is 7.1 percent per annum.",
    "Senior citizens receive an additional 0.5 percent interest on all Fixed Deposit tenures.",
    "Fixed Deposit nomination facility is available for all account holders.",
]

# a query written completely differently from the matching document's wording
# -- this is exactly the vocabulary-mismatch case BM25 cannot solve
DENSE_QUERY = "I want to close my deposit early, will I lose money on my returns?"


def build_dense_vectors(texts: list, n_components: int = 5):
    """
    Turns a list of texts into dense vectors:
      1. TfidfVectorizer -> sparse vectors (mostly zeros)
      2. TruncatedSVD     -> compresses into a small dense vector
    Returns: (dense_vectors, fitted_vectorizer, fitted_svd)
    """
    vectorizer = TfidfVectorizer()
    sparse_matrix = vectorizer.fit_transform(texts)

    # n_components must be smaller than the number of documents and features
    n_components = min(n_components, sparse_matrix.shape[1] - 1, len(texts) - 1)
    svd = TruncatedSVD(n_components=n_components, random_state=42)
    dense_matrix = svd.fit_transform(sparse_matrix)

    return dense_matrix, vectorizer, svd


def normalize_vector(v: np.ndarray) -> np.ndarray:
    """Scales a vector to length 1 -- same purpose as normalize_embeddings=True
    in the real project's model.encode() call."""
    norm = np.linalg.norm(v)
    if norm == 0:
        return v
    return v / norm


def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
    """Standard cosine similarity formula. Works whether or not vectors
    are pre-normalized, but is cheapest when they already are.
    Guards against a zero-length vector (can happen with tiny/degenerate
    corpora) which would otherwise divide by zero and return NaN."""
    denom = np.linalg.norm(a) * np.linalg.norm(b)
    if denom == 0:
        return 0.0
    return float(np.dot(a, b) / denom)


# build dense vectors for our knowledge base
dense_vectors, vectorizer, svd = build_dense_vectors(DENSE_KNOWLEDGE_BASE, n_components=5)

# normalize every stored vector -- mirrors normalize_embeddings=True
dense_vectors_normalized = np.array([normalize_vector(v) for v in dense_vectors])

print("=" * 65)
print("DENSE KNOWLEDGE BASE BUILT (TF-IDF + SVD stand-in)")
print("=" * 65)
for i, text in enumerate(DENSE_KNOWLEDGE_BASE):
    print(f"  Doc {i} | vector shape={dense_vectors_normalized[i].shape} | {text[:60]}...")

print(f"\nEach document is now a dense vector with {dense_vectors_normalized.shape[1]} numbers,")
print("every position non-zero -- this is what 'dense' means, compared")
print("to TF-IDF's mostly-zero sparse vectors from Topic 1.")
print(f"\nTest query: '{DENSE_QUERY}'")
print("\nModule 1 complete. Run Module 2 next.")


DENSE KNOWLEDGE BASE BUILT (TF-IDF + SVD stand-in)
  Doc 0 | vector shape=(5,) | Premature withdrawal of FD incurs a 1 percent penalty on the...
  Doc 1 | vector shape=(5,) | Fixed Deposit premature closure is allowed subject to applic...
  Doc 2 | vector shape=(5,) | Early exit from your deposit account will attract a deductio...
  Doc 3 | vector shape=(5,) | The Fixed Deposit interest rate for 24 months is 7.1 percent...
  Doc 4 | vector shape=(5,) | Senior citizens receive an additional 0.5 percent interest o...
  Doc 5 | vector shape=(5,) | Fixed Deposit nomination facility is available for all accou...

Each document is now a dense vector with 5 numbers,
every position non-zero -- this is what 'dense' means, compared
to TF-IDF's mostly-zero sparse vectors from Topic 1.

Test query: 'I want to close my deposit early, will I lose money on my returns?'

Module 1 complete. Run Module 2 next.


### Module 2: Dense Retrieval Search

Embed the query with the SAME vectorizer/SVD fitted on the knowledge base, then rank documents by cosine similarity.

In [2]:

# ------------------------------------------------------------------
# MODULE 2: Dense Retrieval Search
#
# CRITICAL RULE: the query must be transformed using the SAME
# vectorizer and SAME SVD that were fit on the knowledge base.
# In the real project this rule becomes "use the same embedding
# model at ingest time and query time" -- using a different model
# (or a differently-fit one) makes the vectors incomparable.
# ------------------------------------------------------------------

def embed_query(query: str, vectorizer, svd) -> np.ndarray:
    """Transforms a NEW piece of text using an ALREADY-FITTED
    vectorizer and SVD -- never re-fit on the query alone."""
    sparse_query = vectorizer.transform([query])
    dense_query = svd.transform(sparse_query)[0]
    return normalize_vector(dense_query)


def dense_search(query: str, corpus_texts: list, corpus_vectors: np.ndarray,
                  vectorizer, svd, top_k: int = 3) -> list:
    """Brute-force dense retrieval: embed the query, compare against
    every stored vector, return the top_k closest by cosine similarity.
    This mirrors Chapter 3's in-memory search_knowledge_base() function."""
    query_vector = embed_query(query, vectorizer, svd)
    scores = []
    for i, doc_vector in enumerate(corpus_vectors):
        score = cosine_similarity(query_vector, doc_vector)
        scores.append((score, i))
    scores.sort(key=lambda x: x[0], reverse=True)
    return scores[:top_k]


results = dense_search(DENSE_QUERY, DENSE_KNOWLEDGE_BASE, dense_vectors_normalized,
                        vectorizer, svd, top_k=6)

print("=" * 65)
print("DENSE RETRIEVAL RESULTS (ranked by cosine similarity)")
print("=" * 65)
print(f"Query: '{DENSE_QUERY}'\n")
for rank, (score, idx) in enumerate(results, start=1):
    print(f"  Rank {rank} | score={score:.4f} | {DENSE_KNOWLEDGE_BASE[idx][:65]}...")

print("\nNotice: the query shares almost NO exact words with Doc 2")
print("('Early exit ... deduction from your returns') yet dense retrieval")
print("can still find it as relevant, because the underlying MEANING is close.")
print("A pure keyword search (BM25) would likely miss or underrank Doc 2")
print("since 'close', 'lose money' do not literally appear in it.")
print("\nModule 2 complete. Run Module 3 next.")


DENSE RETRIEVAL RESULTS (ranked by cosine similarity)
Query: 'I want to close my deposit early, will I lose money on my returns?'

  Rank 1 | score=0.8751 | Early exit from your deposit account will attract a deduction fro...
  Rank 2 | score=0.3784 | Fixed Deposit premature closure is allowed subject to applicable ...
  Rank 3 | score=0.3209 | Senior citizens receive an additional 0.5 percent interest on all...
  Rank 4 | score=0.2903 | Premature withdrawal of FD incurs a 1 percent penalty on the appl...
  Rank 5 | score=0.1342 | Fixed Deposit nomination facility is available for all account ho...
  Rank 6 | score=0.0833 | The Fixed Deposit interest rate for 24 months is 7.1 percent per ...

Notice: the query shares almost NO exact words with Doc 2
('Early exit ... deduction from your returns') yet dense retrieval
can still find it as relevant, because the underlying MEANING is close.
A pure keyword search (BM25) would likely miss or underrank Doc 2
since 'close', 'lose money' do not 

### Module 3: Why Normalization Matters

Proves the claim that un-normalized vectors let longer texts win purely by length, regardless of relevance.

In [3]:

# ------------------------------------------------------------------
# MODULE 3: Why Normalization Matters
#
# Without normalization, a longer document can produce a longer
# (larger-magnitude) vector, which can inflate similarity scores
# even when the shorter document is actually MORE relevant.
# ------------------------------------------------------------------

# reuse the un-normalized dense_vectors from Module 1 for comparison
query_vector_raw = svd.transform(vectorizer.transform([DENSE_QUERY]))[0]

print("=" * 65)
print("NORMALIZED vs UN-NORMALIZED COSINE SIMILARITY")
print("=" * 65)
print(f"{'Doc':<5} | {'Un-normalized dot':>18} | {'Cosine (normalized)':>20}")
print("-" * 50)

for i in range(len(DENSE_KNOWLEDGE_BASE)):
    raw_dot = float(np.dot(query_vector_raw, dense_vectors[i]))          # NOT normalized
    cosine_score = cosine_similarity(query_vector_raw, dense_vectors[i])  # always normalizes internally
    print(f"{i:<5} | {raw_dot:>18.4f} | {cosine_score:>20.4f}")

print("\nThe raw dot product column can rank documents differently than")
print("proper cosine similarity, because it is sensitive to each vector's")
print("magnitude (which is influenced by document length and word rarity),")
print("not just its direction (which is what actually encodes meaning).")
print("This is why the real project always calls encode(..., normalize_embeddings=True).")
print("\nModule 3 complete. Run Module 4 next.")


NORMALIZED vs UN-NORMALIZED COSINE SIMILARITY
Doc   |  Un-normalized dot |  Cosine (normalized)
--------------------------------------------------
0     |             0.1115 |               0.2903
1     |             0.1629 |               0.3784
2     |             0.3789 |               0.8751
3     |             0.0321 |               0.0833
4     |             0.1377 |               0.3209
5     |             0.0534 |               0.1342

The raw dot product column can rank documents differently than
proper cosine similarity, because it is sensitive to each vector's
magnitude (which is influenced by document length and word rarity),
not just its direction (which is what actually encodes meaning).
This is why the real project always calls encode(..., normalize_embeddings=True).

Module 3 complete. Run Module 4 next.


### Module 4: The Discrimination-Gap Problem

Rebuilds the project's own measured finding: the similarity gap between 'same topic, different words' and 'different topic entirely' can be dangerously small for short, ambiguous text.

In [4]:

# ------------------------------------------------------------------
# MODULE 4: The Discrimination-Gap Problem
#
# This reproduces the real finding from the project: short, ambiguous
# text can produce a very small gap between "genuinely related" and
# "genuinely unrelated" similarity scores -- making dense retrieval
# alone risky for this kind of corpus.
# ------------------------------------------------------------------

# two short, differently-worded texts about the SAME topic (FD payout delay)
fd_complaint_a = "the maturity amount of my FD was supposed to arrive but nothing came"
fd_complaint_b = "my amount is pending since June please credit it to my bank fast"

# a short text about a COMPLETELY different topic (login issue)
unrelated_email = "app login is not working otp is not coming please help"

comparison_texts = [fd_complaint_a, fd_complaint_b, unrelated_email]
comp_vectors, comp_vectorizer, comp_svd = build_dense_vectors(comparison_texts, n_components=2)
comp_vectors_norm = np.array([normalize_vector(v) for v in comp_vectors])

sim_same_topic = cosine_similarity(comp_vectors_norm[0], comp_vectors_norm[1])
sim_different_topic = cosine_similarity(comp_vectors_norm[0], comp_vectors_norm[2])
gap = sim_same_topic - sim_different_topic

print("=" * 65)
print("DISCRIMINATION GAP -- same topic vs different topic")
print("=" * 65)
print(f"FD complaint A : {fd_complaint_a}")
print(f"FD complaint B : {fd_complaint_b}")
print(f"Unrelated email: {unrelated_email}\n")
print(f"similarity(FD-A, FD-B)          = {sim_same_topic:.4f}   (same topic, different words)")
print(f"similarity(FD-A, unrelated)     = {sim_different_topic:.4f}   (completely different topic)")
print(f"Gap                              = {gap:.4f}")

print("\nWith this small toy example and only a handful of training texts,")
print("the exact gap number will differ from the real project's measured")
print("0.0393 gap (which used the full trained multilingual model on a")
print("much larger corpus) -- but the PATTERN is the same lesson:")
print("short, ambiguous, similarly-worded text produces a thin margin")
print("between 'related' and 'unrelated', which is risky to rely on alone.")
print("\nModule 4 complete. Run Module 5 next.")


DISCRIMINATION GAP -- same topic vs different topic
FD complaint A : the maturity amount of my FD was supposed to arrive but nothing came
FD complaint B : my amount is pending since June please credit it to my bank fast
Unrelated email: app login is not working otp is not coming please help

similarity(FD-A, FD-B)          = 0.7405   (same topic, different words)
similarity(FD-A, unrelated)     = -0.2214   (completely different topic)
Gap                              = 0.9619

With this small toy example and only a handful of training texts,
the exact gap number will differ from the real project's measured
0.0393 gap (which used the full trained multilingual model on a
much larger corpus) -- but the PATTERN is the same lesson:
short, ambiguous, similarly-worded text produces a thin margin
between 'related' and 'unrelated', which is risky to rely on alone.

Module 4 complete. Run Module 5 next.


### Module 5: Exact Identifiers Break Dense Retrieval

Demonstrates why an FD reference number should never be looked up via semantic similarity.

In [5]:

# ------------------------------------------------------------------
# MODULE 5: Exact Identifiers Break Dense Retrieval
#
# Reference numbers like "BJ2019FD7717" get broken into fragments
# during vectorization. A different customer's reference number that
# shares some of those same fragments can end up looking deceptively
# similar -- exactly the cross-customer mix-up risk described in the
# theory. This is why identifiers must use exact-match lookup, never
# semantic search.
# ------------------------------------------------------------------

reference_numbers = [
    "BJ2019FD7717",
    "BJ2021FD4432",   # shares "BJ" and "FD" fragments with the one above
    "XY2019FD7717",   # shares "2019" and "FD7717" fragments
]

# NOTE: the default word-level tokenizer treats each whole identifier as a
# single token, so it cannot see the shared "BJ" / "FD" / "2019" fragments
# that a real subword tokenizer would extract. We use character n-grams
# here instead, so the fragment-sharing behaviour described in the theory
# actually shows up in the numbers.
ref_vectorizer = TfidfVectorizer(analyzer="char", ngram_range=(2, 4))
ref_sparse = ref_vectorizer.fit_transform(reference_numbers)
ref_svd = TruncatedSVD(n_components=2, random_state=42)
ref_vectors = ref_svd.fit_transform(ref_sparse)
ref_vectors_norm = np.array([normalize_vector(v) for v in ref_vectors])

query_ref = "BJ2019FD7717"
query_ref_vector = normalize_vector(
    ref_svd.transform(ref_vectorizer.transform([query_ref]))[0]
)

print("=" * 65)
print("EXACT IDENTIFIER LOOKUP -- semantic search vs exact match")
print("=" * 65)
print(f"Looking up: '{query_ref}' using SEMANTIC similarity\n")
for i, ref in enumerate(reference_numbers):
    score = cosine_similarity(query_ref_vector, ref_vectors_norm[i])
    exact_match = "EXACT MATCH" if ref == query_ref else "different account"
    print(f"  {ref:<15} | similarity={score:.4f} | {exact_match}")

print("\nIf similarity scores for different accounts are close to the exact")
print("match's score, semantic search could return the WRONG customer's")
print("record. The correct approach for identifiers is a direct equality")
print("check (like a SQL lookup by primary key), never similarity search.")
print("\nCorrect approach:")
for ref in reference_numbers:
    is_match = (ref == query_ref)
    print(f"  exact_match('{query_ref}', '{ref}') -> {is_match}")

print("\nModule 5 complete. All Topic 2 modules done.")
print()
print("=" * 65)
print("CHAPTER 7 TOPIC 2 -- KEY POINTS TO REMEMBER")
print("=" * 65)
print("""
  Dense retrieval matches MEANING, not words -- solves BM25's
  vocabulary mismatch problem.

  Always: same model at ingest AND query time, normalize vectors,
  batch-embed instead of looping one at a time.

  Never: use semantic search for exact identifiers (reference numbers,
  account numbers) -- use exact-match lookup instead.

  This project's real measured discrimination gap (same topic vs
  different topic) was only ~0.039 -- dangerously thin for short,
  ambiguous, Hinglish emails.

  Dense retrieval alone is NOT enough for this corpus.
  Next: Topic 4 (Hybrid Search) combines BM25 + Dense to cover
        each other's blind spots.
""")


EXACT IDENTIFIER LOOKUP -- semantic search vs exact match
Looking up: 'BJ2019FD7717' using SEMANTIC similarity

  BJ2019FD7717    | similarity=1.0000 | EXACT MATCH
  BJ2021FD4432    | similarity=0.2051 | different account
  XY2019FD7717    | similarity=0.9804 | different account

If similarity scores for different accounts are close to the exact
match's score, semantic search could return the WRONG customer's
record. The correct approach for identifiers is a direct equality
check (like a SQL lookup by primary key), never similarity search.

Correct approach:
  exact_match('BJ2019FD7717', 'BJ2019FD7717') -> True
  exact_match('BJ2019FD7717', 'BJ2021FD4432') -> False
  exact_match('BJ2019FD7717', 'XY2019FD7717') -> False

Module 5 complete. All Topic 2 modules done.

CHAPTER 7 TOPIC 2 -- KEY POINTS TO REMEMBER

  Dense retrieval matches MEANING, not words -- solves BM25's
  vocabulary mismatch problem.

  Always: same model at ingest AND query time, normalize vectors,
  batch-embed inste